In [63]:
import pandas as pd
import os

### Load MAG info from VMGC paper

In [64]:
sgb_info = pd.read_csv("VMGC_db_build/VMGC_orig_files/VMGC_prokaryote_SGB.info", sep='\t', index_col=0)
sgb_info['taxonomy'] = (
    'd__' + sgb_info['Kingdom'] + ';' +
    'p__' + sgb_info['Phylum'] + ';' +
    'c__' + sgb_info['Class'] + ';' +
    'o__' + sgb_info['Order'] + ';' +
    'f__' + sgb_info['Family'] + ';' +
    'g__' + sgb_info['Genus'] + ';' +
    's__' + sgb_info['Species']
)
sgb_to_tax = sgb_info['taxonomy'].to_dict()

In [65]:
df = pd.read_csv('VMGC_db_build/VMGC_orig_files/VMGC_prokaryote_MAG.info', sep='\t', index_col=0)
df = df[ ['BioSample_ID', 'Collection/isolation_source', 'Type',
       'Genome_size_(bp)', '%_Completeness',
       '%_Contamination', 'Quality_score', 
        'Genome_quality', ]]
midas_info = pd.read_csv('VMGC_db_build/genomes_and_SGBs.csv', index_col=0).drop(columns=['Quality_score','Genome_quality', 'genome_is_VMGC_representative','VMGC_representative'])

df=df.join(midas_info).reset_index()
df['taxonomy'] = df['SGB'].map(sgb_to_tax)


### Group by species

In [49]:
species_df = df.groupby('species_name').agg(
        taxonomy=('taxonomy', 'first'),

        species_code=('species', 'first'),
            representative_genome=('representative', 'first'),

    num_genomes=('Genome_ID', 'count'),
    

    mean_quality_score=('Quality_score', 'mean'),
    std_quality_score=('Quality_score', 'std'),
    range_quality_score=('Quality_score', lambda x: (x.min(), x.max())),


    mean_percent_completeness=('%_Completeness', 'mean'),
    std_percent_completeness=('%_Completeness', 'std'),
    range_percent_completeness=('%_Completeness', lambda x: (x.min(), x.max())),


    mean_percent_contamination=('%_Contamination', 'mean'),
    std_percent_contamination=('%_Contamination', 'std'),
    range_percent_contamination=('%_Contamination', lambda x: (x.min(), x.max())),


    mean_num_bp_per_genome=('Genome_size_(bp)', 'mean'),
    std_num_bp_per_genome=('Genome_size_(bp)', 'std'),
    range_num_bp_per_genome=('Genome_size_(bp)', lambda x: (x.min(), x.max())),

)

species_df

,taxonomy,species_code,representative_genome,num_genomes,mean_quality_score,std_quality_score,range_quality_score,mean_percent_completeness,std_percent_completeness,range_percent_completeness,mean_percent_contamination,std_percent_contamination,range_percent_contamination,mean_num_bp_per_genome,std_num_bp_per_genome,range_num_bp_per_genome
species_name,,,,,,,,,,,,,,,,
Abiotrophia massiliensis,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...,323377,GCF_900604935.1,1,99.1000,NaN,"(99.1, 99.1)",100.0000,NaN,"(100.0, 100.0)",0.180,NaN,"(0.18, 0.18)",1754973.00,NaN,"(1754973, 1754973)"
Acetatifactor sp900066565,d__Bacteria;p__Bacillota_A;c__Clostridia;o__La...,687954,P10710167.mbin.40,1,80.1500,NaN,"(80.15, 80.15)",92.2500,NaN,"(92.25, 92.25)",2.420,NaN,"(2.42, 2.42)",3139699.00,NaN,"(3139699, 3139699)"
Acinetobacter baumannii,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,619896,GCF_000162295.1,5,89.4360,21.670980,"(50.67, 99.25)",90.1360,22.034221,"(50.72, 100.0)",0.140,0.074162,"(0.01, 0.19)",3503209.40,1.028650e+06,"(1672604, 4045681)"
Acinetobacter gyllenbergii,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,219869,GCF_000488195.1,1,91.2800,NaN,"(91.28, 91.28)",99.9800,NaN,"(99.98, 99.98)",1.740,NaN,"(1.74, 1.74)",4551346.00,NaN,"(4551346, 4551346)"
Acinetobacter pseudolwoffii,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,660248,GCF_000369445.1,1,99.5900,NaN,"(99.59, 99.59)",99.9900,NaN,"(99.99, 99.99)",0.080,NaN,"(0.08, 0.08)",3042530.00,NaN,"(3042530, 3042530)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Weizmannia coagulans_A,d__Bacteria;p__Bacillota;c__Bacilli;o__Bacilla...,297050,GCF_001546215.1,1,99.9600,NaN,"(99.96, 99.96)",99.9600,NaN,"(99.96, 99.96)",0.000,NaN,"(0.0, 0.0)",3312432.00,NaN,"(3312432, 3312432)"
Winkia neuii,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,315466,GCF_001546135.1,4,88.0475,20.842748,"(56.79, 98.98)",89.9725,19.788982,"(60.29, 99.98)",0.385,0.229565,"(0.2, 0.7)",2105412.25,4.448660e+05,"(1438379, 2344907)"
Winkia sp002849225,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,743486,GCF_000296485.1,2,76.4750,32.123861,"(53.76, 99.19)",77.9000,31.239978,"(55.81, 99.99)",0.285,0.176777,"(0.16, 0.41)",1754928.00,5.498264e+05,"(1366142, 2143714)"


In [50]:
vmgc_species_to_id = species_df['species_code'].to_dict()

### Select species with at least 5 genomes

In [51]:
species_df = species_df[species_df['num_genomes'] >= 5]
species_df.shape

(179, 16)

In [52]:
pangenome_counts = []
cluster_levels = [75, 80, 85, 90, 95, 99]

for sp in species_df.index:

    v = vmgc_species_to_id[sp]
    cluster_path = f'VMGC_db/pangenomes/{v}/clusters_99_info.tsv'
    if not os.path.exists(cluster_path):
        print(v)
        continue

    vmgc_clusters = pd.read_csv(cluster_path, sep='\t')   

    for c in cluster_levels:
        
        vmgc_count = vmgc_clusters[f'centroid_{c}'].unique().shape[0]
        
        pangenome_counts += [[sp, v, c, vmgc_count,]]


/var/folders/nh/_pzp_lps21jf91jy0p779fvm0000gn/T/ipykernel_52325/2162491743.py:12: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  vmgc_clusters = pd.read_csv(cluster_path, sep='\t')


In [53]:
pangenome_counts = pd.DataFrame(pangenome_counts)
pangenome_counts.columns = ['species_name', 'VMGC_species_code', 'cluster_level', 'VMGC_pangenome_size']
pangenome_counts['num_genomes'] = pangenome_counts['species_name'].map(species_df['num_genomes'])
pangenome_counts.to_csv('VMGC_pangenome_counts_min_5_genomes.csv', index=False)

In [54]:
pangenome_counts_piv = pangenome_counts.pivot(index='species_name', columns='cluster_level', values='VMGC_pangenome_size')
pangenome_counts_piv.columns = [f'num_genes_in_C{i}_pangenome' for i in pangenome_counts_piv.columns]

In [55]:
pangenome_counts_piv = pangenome_counts_piv[pangenome_counts_piv.columns[::-1]]
pangenome_counts_piv.head()

,num_genes_in_C99_pangenome,num_genes_in_C95_pangenome,num_genes_in_C90_pangenome,num_genes_in_C85_pangenome,num_genes_in_C80_pangenome,num_genes_in_C75_pangenome
species_name,,,,,,
Acinetobacter baumannii,7549,4939,4691,4635,4592,4559
Actinobaculum massiliense,1899,1889,1886,1884,1880,1875
Actinotignum sanguinis,2643,2282,2112,2043,2004,1984
Aerococcus christensenii,8274,3451,2933,2738,2642,2555
Alistipes putredinis,4064,2944,2838,2806,2781,2758


In [56]:
species_df = species_df.join(pangenome_counts_piv)

In [69]:
species_df = species_df[['taxonomy', 'species_code', 'representative_genome', 'num_genomes',
      
       'mean_percent_completeness', 'std_percent_completeness',
       'range_percent_completeness', 'mean_percent_contamination',
       'std_percent_contamination', 'range_percent_contamination',
        'mean_quality_score', 'std_quality_score', 'range_quality_score',
       'mean_num_bp_per_genome', 'std_num_bp_per_genome',
       'range_num_bp_per_genome', 'num_genes_in_C99_pangenome',
       'num_genes_in_C95_pangenome', 'num_genes_in_C90_pangenome',
       'num_genes_in_C85_pangenome', 'num_genes_in_C80_pangenome',
       'num_genes_in_C75_pangenome']]

In [70]:
species_df.to_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/tables/Table_S1A.csv')

In [71]:
species_df

,taxonomy,species_code,representative_genome,num_genomes,mean_percent_completeness,std_percent_completeness,range_percent_completeness,mean_percent_contamination,std_percent_contamination,range_percent_contamination,...,range_quality_score,mean_num_bp_per_genome,std_num_bp_per_genome,range_num_bp_per_genome,num_genes_in_C99_pangenome,num_genes_in_C95_pangenome,num_genes_in_C90_pangenome,num_genes_in_C85_pangenome,num_genes_in_C80_pangenome,num_genes_in_C75_pangenome
species_name,,,,,,,,,,,,,,,,,,,,,
Acinetobacter baumannii,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,619896,GCF_000162295.1,5,90.136000,22.034221,"(50.72, 100.0)",0.140000,0.074162,"(0.01, 0.19)",...,"(50.67, 99.25)",3.503209e+06,1.028650e+06,"(1672604, 4045681)",7549,4939,4691,4635,4592,4559
Actinobaculum massiliense,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,629828,GCF_000315465.1,5,80.300000,18.247766,"(58.14, 98.49)",1.302000,0.944442,"(0.16, 2.75)",...,"(53.09, 93.67)",1.659384e+06,3.788633e+05,"(1272294, 2071203)",1899,1889,1886,1884,1880,1875
Actinotignum sanguinis,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,858790,SRR17635715.sbin.92,5,86.468000,11.984729,"(72.59, 98.85)",0.884000,1.180436,"(0.09, 2.9)",...,"(62.83, 98.4)",1.842602e+06,2.571240e+05,"(1525935, 2154979)",2643,2282,2112,2043,2004,1984
Aerococcus christensenii,d__Bacteria;p__Bacillota;c__Bacilli;o__Lactoba...,542621,P10709641.mbin.2,260,85.814500,11.683608,"(52.76, 99.97)",0.935269,0.840510,"(0.04, 4.74)",...,"(50.17, 99.03)",1.295191e+06,1.911237e+05,"(770800, 1710611)",8274,3451,2933,2738,2642,2555
Alistipes putredinis,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,712851,SAMN00040503.mbin.1,5,89.836000,11.093319,"(76.34, 99.99)",1.464000,1.811555,"(0.41, 4.68)",...,"(52.94, 97.89)",2.021397e+06,2.853255e+05,"(1659934, 2321816)",4064,2944,2838,2806,2781,2758
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Varibaculum massiliense,d__Bacteria;p__Actinomycetota;c__Actinomycetia...,904423,ERR10897914.mbin.4,12,87.870000,8.595570,"(77.09, 99.99)",1.257500,0.735083,"(0.23, 2.94)",...,"(66.37, 98.57)",1.776246e+06,2.250029e+05,"(1457354, 2101645)",9558,4197,2990,2643,2550,2497
Veillonella montpellierensis,d__Bacteria;p__Bacillota_C;c__Negativicutes;o_...,674615,SRR16470998.sbin.6,39,74.209744,15.710640,"(52.36, 99.99)",0.552821,0.669233,"(0.0, 2.18)",...,"(51.41, 97.04)",1.333022e+06,3.377077e+05,"(781301, 1976838)",12344,3756,3149,2979,2902,2853
Veillonella parvula_A,d__Bacteria;p__Bacillota_C;c__Negativicutes;o_...,773494,GCF_000215025.2,6,93.465000,8.993808,"(78.14, 100.0)",0.803333,1.529898,"(0.08, 3.92)",...,"(58.54, 99.44)",1.902467e+06,2.154803e+05,"(1574768, 2152140)",6875,3504,2660,2468,2410,2374


In [72]:
species_df.columns

Index(['taxonomy', 'species_code', 'representative_genome', 'num_genomes',
       'mean_percent_completeness', 'std_percent_completeness',
       'range_percent_completeness', 'mean_percent_contamination',
       'std_percent_contamination', 'range_percent_contamination',
       'mean_quality_score', 'std_quality_score', 'range_quality_score',
       'mean_num_bp_per_genome', 'std_num_bp_per_genome',
       'range_num_bp_per_genome', 'num_genes_in_C99_pangenome',
       'num_genes_in_C95_pangenome', 'num_genes_in_C90_pangenome',
       'num_genes_in_C85_pangenome', 'num_genes_in_C80_pangenome',
       'num_genes_in_C75_pangenome'],
      dtype='object')